# Laboratory 5B — How do we locate the quasi-harmonic equilibrium?

At each temperature, Quantas builds a free-energy curve $F(V,T)$ from the same
seven first-principles states. We now compare two ways of representing that
curve:

- a third-degree polynomial;
- a third-order Birch–Murnaghan EOS (BM3).

The underlying CRYSTAL data are identical. Only the numerical representation of
the free-energy minimum changes.


## 1. Run two ambient-pressure QHA models

We deliberately calculate a dense temperature section at $P=0$ GPa. The
calculation itself is inexpensive because the costly phonon data are already
available.


In [ ]:
!quantas qha run mgo_qha.yaml \
    --scheme freq \
    --minimization poly \
    --thermal-expansion mixed_derivative \
    --temperature 0 1500 50 \
    --pressure 0 0 1 \
    --energy-degree 3 \
    --frequency-degree 3 \
    --output mgo_qha_poly_0GPa.hdf5 \
    --report mgo_qha_poly_0GPa.log \
    --no-progress --force


In [ ]:
!quantas qha run mgo_qha.yaml \
    --scheme freq \
    --minimization eos \
    --eos BM3 \
    --thermal-expansion mixed_derivative \
    --temperature 0 1500 50 \
    --pressure 0 0 1 \
    --energy-degree 3 \
    --frequency-degree 3 \
    --output mgo_qha_bm3_0GPa.hdf5 \
    --report mgo_qha_bm3_0GPa.log \
    --no-progress --force


In [ ]:
import pandas as pd
from mineral_science_tools.qha_lab import method_comparison

method_comparison(
    "mgo_qha_poly_0GPa.hdf5",
    "mgo_qha_bm3_0GPa.hdf5",
    temperature=300.0,
    pressure=0.0,
)


### Question 1

Which property changes least between the two minimization methods? Which of
$V$, $K_T$ and $\alpha_V$ appears more sensitive?

Why are derivatives/curvatures generally more sensitive to the fitted
free-energy shape than the position of its minimum?

**Your answer:**  
_Write 4–5 lines here._


## 2. Absolute volume: model accuracy becomes visible


In [ ]:
from mineral_science_tools.qha_lab import plot_absolute_volume
exp_volume = pd.read_csv("data/mgo_reeber1995_volume_expansion.csv")
exp_volume = exp_volume[exp_volume["T (K)"] <= 1500]

plot_absolute_volume(
    "mgo_qha_poly_0GPa.hdf5",
    "mgo_qha_bm3_0GPa.hdf5",
    exp_volume,
);


### Question 2

Is the disagreement mainly a random scatter, or a systematic offset in the
absolute volume?

The calculations use PBE. What does a systematic offset tell you about the
underlying first-principles model rather than about q-point convergence?

**Your answer:**  
_Write 3–4 lines here._


## 3. Normalize the volume before judging the thermal trend


In [ ]:
from mineral_science_tools.qha_lab import plot_normalized_volume
plot_normalized_volume(
    "mgo_qha_poly_0GPa.hdf5",
    "mgo_qha_bm3_0GPa.hdf5",
    exp_volume,
    reference_temperature=300.0,
);


### Question 3 — Absolute value versus behaviour

Compare the absolute-volume and normalized-volume figures.

Can a model that overestimates the absolute lattice volume still reproduce the
**thermal expansion trend** well? Why is this distinction important in
computational mineralogy?

**Your answer:**  
_Write 5–6 lines here._


## 4. Which minimization will we carry forward?

For the rest of the teaching workflow we will use the polynomial result as the
primary model and retain BM3 as a sensitivity check. This is not a universal
ranking: both are legitimate representations when their fits are well behaved
and the minimum stays inside the sampled volume range.

The next notebook compares the chosen QHA result with independent experimental
thermodynamic, structural and elastic data.
